# Work with Measurement Service

This notebook shows what can be done with the Measurement Service.

In [1]:
%cd ../
%load_ext autoreload
%autoreload 2

/Users/matthaei/Documents/code/python/bachelor-project


In [2]:
MIGRATE_DATABASE = True

In [3]:
from src.measurements.measurement_service import MeasurementService
from src.weather_stations.weather_station_service import WeatherStationService
from src.database.database_service import DatabaseService
from omegaconf import DictConfig, OmegaConf
from hydra import compose, initialize_config_dir
import os
from datetime import datetime

In [5]:
# Initialize Hydra configuration
config_dir = os.path.abspath("./conf")

# Initialize Hydra with the config directory
with initialize_config_dir(config_dir=config_dir, version_base=None):
    cfg = compose(config_name="config")

In [6]:
database_service = DatabaseService(cfg)

if MIGRATE_DATABASE:
    database_service.create_tables()

2025-09-04 17:20:15.965 | INFO     | src.database.database_service:create_tables:22 - Tables created


In [7]:
weather_station_service = WeatherStationService(cfg, database_service)
weather_stations_df = weather_station_service.load_from_database()

2025-09-04 17:20:28.134 | INFO     | src.weather_stations.weather_station_data_provider:load_from_database:225 - Loaded 26 weather stations from database


## Fill data from DWD

In [8]:
measurement_service = MeasurementService(cfg, database_service, weather_stations_df)

In [15]:
df = measurement_service.fill_database_with_measurements(only_now=False)

2025-09-04 17:27:47.247 | INFO     | src.measurements.measurement_data_provider:_download_file:323 - Start downloading file from https://opendata.dwd.de/climate_environment/CDC/observations_germany/climate/10_minutes/wind/historical/10minutenwerte_wind_00096_20200101_20241231_hist.zip
2025-09-04 17:27:47.750 | INFO     | src.measurements.measurement_data_provider:_download_file:345 - Downloaded file from https://opendata.dwd.de/climate_environment/CDC/observations_germany/climate/10_minutes/wind/historical/10minutenwerte_wind_00096_20200101_20241231_hist.zip as dataframe with 263088 rows
2025-09-04 17:27:47.752 | INFO     | src.measurements.measurement_data_provider:_download_dataset_for_station:294 - Successfully downloaded and processed wind data from https://opendata.dwd.de/climate_environment/CDC/observations_germany/climate/10_minutes/wind/historical/10minutenwerte_wind_00096_20200101_20241231_hist.zip
2025-09-04 17:27:47.752 | INFO     | src.measurements.measurement_data_provider

## Load for datetime

In [16]:
test_time = datetime(2025, 9, 4, 10, 0)
df = measurement_service.load_measurements_from_database_for_datetime(test_time)

2025-09-04 16:21:05.205 | INFO     | src.measurements.measurement_data_provider:load_measurements_from_database_for_datetime:200 - Loaded 17 measurements from database for datetime 2025-09-04 10:00:00


In [18]:
df['station_id'].unique()

array([  96,  164,  303,  427,  880, 1001, 1869, 2794, 2856, 3015, 3158,
       3376, 3987, 5546, 5825, 6253])